<a href="https://colab.research.google.com/github/JosephAFerguson/-UserInterface-Proj2/blob/main/DeepLearningJF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json
from datetime import datetime, timedelta

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [37]:
np.random.seed(0)
torch.manual_seed(0)

t = np.linspace(0, 100, 1000)
data = np.sin(t)

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)\

seq_length = 10
X, y = create_sequences(data, seq_length)


trainX = torch.tensor(X[:, :, None], dtype=torch.float32)
trainY = torch.tensor(y[:, None], dtype=torch.float32)

tensor([[ 8.4201e-01],
        [ 8.9171e-01],
        [ 9.3247e-01],
        [ 9.6391e-01],
        [ 9.8569e-01],
        [ 9.9760e-01],
        [ 9.9953e-01],
        [ 9.9144e-01],
        [ 9.7344e-01],
        [ 9.4568e-01],
        [ 9.0846e-01],
        [ 8.6215e-01],
        [ 8.0720e-01],
        [ 7.4417e-01],
        [ 6.7369e-01],
        [ 5.9647e-01],
        [ 5.1327e-01],
        [ 4.2493e-01],
        [ 3.3235e-01],
        [ 2.3643e-01],
        [ 1.3815e-01],
        [ 3.8480e-02],
        [-6.1572e-02],
        [-1.6101e-01],
        [-2.5883e-01],
        [-3.5406e-01],
        [-4.4575e-01],
        [-5.3297e-01],
        [-6.1486e-01],
        [-6.9059e-01],
        [-7.5941e-01],
        [-8.2063e-01],
        [-8.7363e-01],
        [-9.1788e-01],
        [-9.5295e-01],
        [-9.7847e-01],
        [-9.9420e-01],
        [-9.9997e-01],
        [-9.9573e-01],
        [-9.8153e-01],
        [-9.5749e-01],
        [-9.2387e-01],
        [-8.8100e-01],
        [-8

In [30]:
class CryptoEndpoint:
    listingsEndpoint = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
    latestQuotes = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/latest"
    historicalQuotes = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/historical"

    def __init__(self, apikey) -> None:
        self.headers = {
            'Accepts': 'application/json',
            'X-CMC_PRO_API_KEY': apikey,
        }
        self.coinsInfo = {}
        self.coinsIds = []

    def GetCoinIdentifiers(self):
        session = Session()
        session.headers.update(self.headers)
        response = session.get(
            url=self.listingsEndpoint,
            params={
                "limit": 10,
                "price_min": 1,
                "price_max" : 2
            }
        )

        data = json.loads(response.text)

        for coin in data.get("data", []):
            self.coinsInfo[coin["name"]] = coin["id"]
            self.coinsIds.append(coin["id"])

        print(f"Loaded {len(self.coinsInfo)} coins.")
        return (self.coinsInfo, self.coinsIds)

    def GetCoinLatestPrices(self, coin_id):

        session = Session()
        session.headers.update(self.headers)
        response = session.get(url=self.latestQuotes, params={"id": coin_id})
        data = json.loads(response.text)

        prices = {}
        for coin_id, info in data.get("data", {}).items():
            quote = info["quote"]["USD"]["price"]
            prices[coin_id] = quote

        return prices

    def GetSampleCoinHistoricalData(self, coin_id, days=4):
        session = Session()
        session.headers.update(self.headers)

        end_time = datetime.utcnow()
        start_time = end_time - timedelta(days=days)

        prices = {}

        params = {
            "id": coin_id,
            "time_start": start_time.isoformat(),
            "time_end": end_time.isoformat(),
            "interval": "24h",
        }

        response = session.get(url=self.historicalQuotes, params=params)
        data = json.loads(response.text)
        print(data["status"]["error_code"])
        coin_data = data.get("data", {})

        historicals = []

        for quote in coin_data["quotes"]:
            date = quote.get("timestamp") or quote.get("time_open")
            price = quote["quote"]["USD"]["price"]
            volume = quote["quote"]["USD"]["volume_24h"]
            historicals.append([price,volume])

        returnData = {coin_id: historicals}
        return returnData

In [34]:
ce = CryptoEndpoint(input("Enter API-KEY"))
(coinsInfo, coinsIds) = ce.GetCoinIdentifiers()
data = []
for coinId in coinsIds:
  coinSample = ce.GetSampleCoinHistoricalData(coinId)
  if len(list(coinSample.values())[0]) < 1:
    continue
  data.append(coinSample)
print(coinsInfo)
print(data)

dataX = [list(coinData.values()) for coinData in data]
dataY = [list(coinData.keys()) for coinData in data]

Enter API-KEYa76bd6fc-2b66-4a25-843f-321def3437bd
Loaded 10 coins.


/tmp/ipython-input-613701782.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


0
0
0
0
0
0
0
0
0
0
{'USDC': 3408, 'Toncoin': 11419, 'NEAR Protocol': 6535, 'Worldcoin': 13502, 'Arbitrum': 11841, 'Render': 5690, 'PancakeSwap': 7186, 'Optimism': 11840, 'Lido DAO': 8000, 'Helium': 5665}
[{3408: [[1.0006082439231117, 20063643268.52], [0.9999067648425628, 12886744490.48], [1.0001575020545748, 9592753102.24]]}, {11419: [[2.106541887030274, 188986381.25], [2.0921124789971084, 138819071.19], [2.1300028162543434, 101951978.13]]}, {6535: [[2.779493289156971, 1339083819.44], [2.8787756711801737, 1503886945.39], [2.924967768051715, 1008499779.45]]}, {13502: [[0.859314755000748, 275628954], [0.8098064362389464, 227826900.28], [0.817974354427579, 139119094.23]]}, {11841: [[0.3002274675776444, 284319925.47], [0.2923707096382341, 238990891.15], [0.29973089755310256, 138560873.4]]}, {5690: [[2.6949531729882437, 220274931.23], [2.322547712472358, 182001450.05], [2.423148439182687, 80844224.32]]}, {7186: [[2.4679917389912136, 147201613.81], [2.5327146839662493, 146472652.07], [2.540

In [32]:
#sample ltsm from geeks for geeks
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(
                0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(
                0), self.hidden_dim).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # Take last time step
        return out, hn, cn

In [40]:
trainX = torch.tensor(dataX[:, :, None], dtype=torch.float32)
trainY = torch.tensor(dataY[:, None], dtype=torch.float32)
model = LSTMModel(input_dim=1, hidden_dim=100, layer_dim=1, output_dim=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

TypeError: list indices must be integers or slices, not tuple

In [39]:
num_epochs = 100
h0, c0 = None, None

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs, h0, c0 = model(dataX, h0, c0)

    loss = criterion(outputs, dataY)
    loss.backward()
    optimizer.step()

    h0, c0 = h0.detach(), c0.detach()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

AttributeError: 'list' object has no attribute 'size'